::: {.sb-lesson-links}
[Exercises →](../exercises/ridge-vs-lasso.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/03-regularization.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-regularization.ipynb){download="lesson-regularization.ipynb"}  
Data: [gene_expression_regression.csv](../downloads/data/gene_expression_regression.csv){download="gene_expression_regression.csv"} · [gene_expression_true_genes.csv](../downloads/data/gene_expression_true_genes.csv){download="gene_expression_true_genes.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

## What will you learn in this course? 🧐🧐

This course teaches you two variations of the linear regression model, called regularized regressions.

* What is regularization?
* Ridge model
    * What is a cost/loss function?
    * Bias vs Variance
    * Ridge model penalty
    * Special case of the intercept
* Lasso model
     * Why does the linear model fail?
     * How can we overcome this difficulty?
         * Variable selection problem
     * The Lasso model
         * Intuition
         * Lasso estimator
* Underfitting and overfitting

In [ ]:
import plotly.io as pio
import plotly.graph_objects as go
pio.templates["course"] = go.layout.Template(layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"])
pio.templates.default = "plotly_white+course"
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import make_regression

## What is regularization? 👮

Regularization is a principle widely used in statistics and machine learning. It constrains models to adopt certain behaviours. Here, it lets us:

* reduce the importance of explanatory variables that do not bring relevant information to the model, or remove them altogether;
* deal with problems where there are many more explanatory variables than observations.

Linear models are popular for estimating a continuous target variable $Y$ that depends on explanatory variables $X_{1}, X_{2}, ... , X_{p}$. In general, the number of observations (or samples, often noted $n$) is much greater than the number of explanatory variables $p$. Sometimes, however, it is the opposite: there are many more explanatory variables than observations. Statistical genetics is a good example: time and money constraints rarely let researchers sequence the DNA of more than a thousand individuals, while our DNA contains about 20,000 genes! Classical linear models are not well suited to such high-dimensional problems. In this course, we will see variations of linear models that can tackle this difficulty.


## Ridge model - `L2` Regularization 🌉

### What is a cost function? 💸



::: {.callout-tip title="Intuition"}
Think of the cost function as a score of "how unhappy" the model is with its own predictions. Each prediction that misses the true value adds a penalty, and big misses count much more than small ones (because errors are squared). Fitting the model simply means turning the knobs (the coefficients) until this unhappiness is as small as possible.
:::



The **cost function** is a concept you will use again and again: it is everywhere in machine learning. It is the mathematical quantity we want to minimize when we fit a statistical model. In classical multiple linear regression, the cost function is:

$$
||y-X^{t}\beta||^{2}_{2}
$$

This is the squared Euclidean norm of the error vector (the Euclidean norm of a vector is the square root of the sum of its squared components). In other words, it is the sum of the squared errors, which is the Mean Squared Error multiplied by $n$. The estimator $\beta$, which gives the coefficients of the model, is the vector that minimizes this cost function. Mathematically:

$$
\beta^* = argmin_{\beta}(||y-X^{t}\beta||^{2}_{2})
$$

This cost function gives an estimator with no bias. But its predictions can have a high variance, especially when there are many explanatory variables. Sometimes, a better compromise can be found.

This is why we introduce the notion of **penalty**: a modification of the cost function that controls the **bias vs. variance** trade-off.

### See it with data 🔬: the cost function is a bowl

Let's make the cost function concrete. We generate 30 points that follow $y = 2x + 1$ plus some noise, then we try many candidate slopes (keeping the intercept at 1) and compute the sum of squared errors for each of them. The best slope is simply the bottom of the curve.

In [ ]:
rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 30)
y = 2 * x + 1 + rng.normal(0, 2, 30)   # true slope = 2

slopes = np.linspace(0, 4, 81)
cost = [np.sum((y - (s * x + 1)) ** 2) for s in slopes]   # sum of squared errors

best = slopes[np.argmin(cost)]
print(f"Cost with slope 0: {cost[0]:.0f}")
print(f"Lowest cost: {min(cost):.0f}, reached for slope = {best:.2f}")

fig = px.line(x=slopes, y=cost, title="Cost function of a simple linear regression",
              labels={"x": "Slope tried (β1)", "y": "Sum of squared errors"})
fig.add_vline(x=best, line_dash="dash", annotation_text=f"minimum: β1 = {best:.2f}")
fig.show()

Cost with slope 0: 5094
Lowest cost: 109, reached for slope = 2.10


**What we see:** the cost is huge for bad slopes (about 5,094 with a slope of 0) and drops to about 109 at the bottom of the "bowl", reached for a slope of 2.10, very close to the true slope of 2. The tiny gap comes from the noise in the data. Fitting a linear regression is nothing more than finding the bottom of this bowl (in many dimensions at once).

### Bias vs Variance

Bias and variance are two key notions in statistics, and especially in machine learning when we try to make predictions.



::: {.callout-tip title="Intuition"}
Picture a darts player. **Bias** is when all the darts land in a tight group, but away from the bullseye: the player is consistently wrong in the same direction. **Variance** is when the darts land all over the board: on average they may be centred, but each throw is unpredictable. A model trained on a different sample of data is like a new throw: a simple model is often biased, a very flexible one has a high variance.
:::



Let's consider a problem where $Y$ is the target variable, $X$ is the matrix of explanatory variables and $\epsilon$ is an error term with mean 0 and variance $\sigma^2$.

We want to model $Y$ using the explanatory variables $X$. We assume that there is a function $f$ that represents the true relationship between $Y$ and $X$, such that:

$$
Y=f(X)+\epsilon
$$

At the end of our work as data scientists, we obtain an estimate $\widehat{f}$ of the function $f$. The mean squared error (often noted MSE) of this estimate can then be split into a bias term (the average deviation from the true function) and a variance term (how much the estimator spreads around its own mean). This decomposition is written as follows:

$$
E[(Y-\widehat{f})^2]=E[f-\widehat{f}]^2+E[\hat{f}^2]-E[\widehat{f}]^2+\sigma^2
$$

$$
E[(Y-\widehat{f})^2]=bias^2+variance+\sigma^2
$$

The bias measures how far the average prediction is from the true function $f$, the variance measures how much the predictions vary, and $\sigma^2$ is the part of $Y$ that no model can explain, also called noise.

### See it with data 🔬: bias and variance of two models

To *see* bias and variance, we need many training sets. Since we only have one world, we cheat with synthetic data: the true function is $f(x) = \sin(1.5x)$, and we draw 100 different noisy samples of 20 points. On each sample, we fit a straight line (degree 1, a simple model), a degree-3 polynomial and a degree-10 polynomial (a very flexible model). Then we measure how far the *average* prediction is from $f$ (bias²) and how much the predictions spread around their average (variance).

In [ ]:
def true_f(x):
    return np.sin(1.5 * x)

x_grid = np.linspace(0.1, 2.9, 60)
preds = {1: [], 3: [], 10: []}          # polynomial degree -> list of predicted curves
for i in range(100):                     # 100 different training sets
    r = np.random.default_rng(i)
    xs = r.uniform(0, 3, 20)
    ys = true_f(xs) + r.normal(0, 0.3, 20)
    for degree in preds:
        model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
        preds[degree].append(model.fit(xs.reshape(-1, 1), ys).predict(x_grid.reshape(-1, 1)))

summary = pd.DataFrame({
    "bias²": {d: np.mean((np.mean(p, axis=0) - true_f(x_grid)) ** 2) for d, p in preds.items()},
    "variance": {d: np.mean(np.var(p, axis=0)) for d, p in preds.items()}}).round(3)
summary.index.name = "degree"
summary

,bias²,variance
degree,,
1,0.160,0.031
3,0.002,0.021
10,0.120,35.019


In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Degree 1 (simple)", "Degree 10 (flexible)"])
for col, degree in enumerate([1, 10], start=1):
    for curve in preds[degree][:20]:     # draw the first 20 fitted curves
        fig.add_trace(go.Scatter(x=x_grid, y=curve, mode="lines", line=dict(color="#999999", width=1),
                                 showlegend=False), row=1, col=col)
    fig.add_trace(go.Scatter(x=x_grid, y=true_f(x_grid), mode="lines", line=dict(color="#E4572E", width=3),
                             name="true f(x)", showlegend=(col == 1)), row=1, col=col)
fig.update_yaxes(range=[-2, 2], title_text="prediction")
fig.update_xaxes(title_text="x")
fig.update_layout(title="20 models trained on 20 different samples")
fig.show()

**What we see:** the straight lines (left) all look alike, so their variance is tiny (0.031), but they all miss the curve in the same way: their bias² is 0.16. The degree-10 polynomials (right) can follow the curve, but they change wildly from one sample to the next: their variance explodes to about 35. The degree-3 model is the sweet spot here, with both a small bias² (0.002) and a small variance (0.021). This is the bias vs. variance trade-off, and regularization is one way to control it.

### Ridge model penalty



::: {.callout-tip title="Intuition"}
Imagine each coefficient is attached to zero by a rubber band. Without the band (ordinary regression), coefficients can grow as large as they want to fit every detail of the training data, including the noise. The Ridge penalty is the rubber band: the larger $\lambda$, the stronger it pulls all the coefficients towards zero. A coefficient only stays large if the variable really earns its place by improving the fit a lot.
:::



The Ridge model is a penalized version of the multiple linear model whose cost function is:

$$
||y-X^{t}\beta||^{2}_{2}+\lambda||\beta||^{2}_{2},\lambda\in\mathbb{R^{+*}}
$$ 

With this penalty, changing the parameters $\beta$ may improve the fit (the first term), but larger parameters also increase the second term, the penalty. So the model has to make choices: it gives large parameters only to the explanatory variables that carry the most relevant information about the target variable, and keeps the parameters of less relevant variables close to zero.

In terms of bias vs. variance trade-off, the Ridge model behaves as follows:

* $\lambda = 0$ corresponds to the ordinary linear model, with maximal variance (and the closest fit to the training data)
* Bias increases as $\lambda$ increases
* Variance decreases as $\lambda$ increases
* $\lambda = \infty$ corresponds to the model where all the parameters are zero, hence $\beta_i = 0 \forall i \neq 0$ and the estimator is $\bar{Y}$ (the average of the $Y$ values). This is the model with the lowest possible variance.

### See it with data 🔬: Ridge shrinks the coefficients as λ grows

Let's watch the "rubber band" at work. We create a dataset with 6 explanatory variables, of which only 3 truly influence the target (the true coefficients of the 3 others are 0). We then fit a Ridge model for many values of λ (called `alpha` in scikit-learn) and draw each coefficient as a function of λ.

In [ ]:
X, y, true_coef = make_regression(n_samples=60, n_features=6, n_informative=3, noise=10,
                                  coef=True, random_state=1)
print("True coefficients:", true_coef.round(1))

alphas = np.logspace(-2, 4, 40)       # from 0.01 to 10,000
paths = pd.DataFrame([Ridge(alpha=a).fit(X, y).coef_ for a in alphas],
                     index=alphas, columns=[f"X{j}" for j in range(6)])

fig = px.line(paths, log_x=True, title="Ridge coefficient path",
              labels={"index": "λ (alpha, log scale)", "value": "coefficient", "variable": "variable"})
fig.show()
paths.iloc[[0, 20, 30, 39]].round(1)   # a few values of λ

True coefficients: [ 0.  81.9  0.  98.2  0.  31.1]


,X0,X1,X2,X3,X4,X5
0.010000,0.2,82.0,0.9,98.1,0.6,33.8
11.937766,1.4,67.8,3.0,81.5,-3.8,27.6
412.462638,1.2,10.5,1.8,12.6,-3.0,4.1
10000.000000,0.1,0.5,0.1,0.6,-0.2,0.2


**What we see:** with a tiny λ (0.01), Ridge gives the same coefficients as ordinary least squares: about 82, 98 and 34 for the three useful variables, and values close to 0 for the others. As λ grows, *all* coefficients shrink smoothly towards 0 (around λ = 10,000 the largest is only about 0.6), but none of them becomes exactly 0. Notice that the useless variables never become exactly 0 either: Ridge does not select variables.

What happens when λ becomes huge? The bullet list above says the model ends up predicting $\bar{Y}$. Let's check it, and see at the same time that the intercept is not penalized:

In [ ]:
huge = Ridge(alpha=1e8).fit(X, y)
print("Coefficients:", huge.coef_.round(4))
print(f"Intercept: {huge.intercept_:.3f}   |   mean of y: {y.mean():.3f}")
print("First 3 predictions:", huge.predict(X[:3]).round(3))

Coefficients: [ 0.      0.0001  0.      0.0001 -0.      0.    ]
Intercept: 16.787   |   mean of y: 16.787
First 3 predictions: [16.787 16.787 16.787]


**What we see:** with λ = 100,000,000, all the coefficients are practically 0, but the intercept is still free: it equals the mean of $y$ (both are printed above and they match). So every prediction is just $\bar{Y}$, the lowest-variance model possible. If the intercept were penalized too, the model would predict 0 for everyone, which makes no sense.

### Special case of the intercept

The intercept is the parameter $\beta_0$ of the model. It is not associated with any explanatory variable: it is the estimated mean level of $Y$ when all the explanatory variables are 0. That is why, in practice, it is not penalized. The cost function of the Ridge model with intercept is therefore:

$$
||y-\beta_0-X^{t}\beta||^{2}_{2}+\lambda||\beta||^{2}_{2},\lambda\in\mathbb{R^{+*}}
$$

## Lasso model - `L1` Regularization 🪢

### Why does the linear model fail?

As we saw earlier, the linear model is written as follows:

$$
Y_i = \beta_0 + X_{i,1}\beta_1 + ... + X_{i,p}\beta_p + \epsilon_i \forall_i \in[\![1,n]\!]
$$

Or in matrix form:

$$
Y=X^{t}\beta+\epsilon
$$



::: {.callout-tip title="Intuition"}
Think of a simple puzzle: "$a + b = 10$, find $a$ and $b$". There is one equation and two unknowns, so $a=5, b=5$ works, but so does $a=100, b=-90$. With more explanatory variables than observations, linear regression faces the same puzzle: many different sets of coefficients fit the training data *perfectly*, and nothing tells the model which one is the right one.
:::



If $p > n$, the first form gives $n$ equations with $p + 1$ unknown parameters $\beta_0,\beta_1,...,\beta_p$. But a system with more unknowns than independent equations is underdetermined: it has an infinite number of solutions.

In matrix form, $X^{t}X$ is a $p*p$ matrix, but its rows and columns are linear combinations of only $n$ vectors. So its rank is at most ***n*** $< p$: the matrix is not invertible, and the parameters cannot be estimated.

### See it with data 🔬: when there are more variables than observations

The file `gene_expression_regression.csv` contains the expression level of 2,000 genes for 801 patients, plus a continuous `target`. We keep only **200 patients for training**, so we have $p = 2000 > n = 200$, and the other 601 patients as a test set.

In [ ]:
genes = pd.read_csv("../data/gene_expression_regression.csv")
X_genes, y_genes = genes.drop(columns="target"), genes["target"]
X_train, X_test, y_train, y_test = train_test_split(X_genes, y_genes, train_size=200, random_state=0)
print("Training set:", X_train.shape, "  Test set:", X_test.shape)

# X^t X is a 2000 x 2000 matrix... but what is its rank?
print("Rank of X^t X:", np.linalg.matrix_rank(X_train.T @ X_train))

Training set: (200, 2000)   Test set: (601, 2000)


Rank of X^t X: 200


The matrix $X^tX$ has 2,000 rows and columns, but its rank is only 200: it cannot be inverted, exactly as explained above. scikit-learn still returns *one* of the infinitely many solutions. Let's see how good it is, and compare it with a Ridge model:

In [ ]:
models = {
    "Linear regression": LinearRegression(),
    "Ridge (alpha=100)": make_pipeline(StandardScaler(), Ridge(alpha=100)),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    rows.append({"model": name,
                 "R² train": r2_score(y_train, model.predict(X_train)),
                 "R² test": r2_score(y_test, model.predict(X_test))})
pd.DataFrame(rows).round(3)

,model,R² train,R² test
0,Linear regression,1.000,0.409
1,Ridge (alpha=100),0.994,0.509


**What we see:** the linear regression gets a *perfect* R² of 1.0 on the training set: with 2,000 coefficients for 200 patients, it can fit every point exactly, noise included. But on new patients its R² falls to 0.409. The Ridge model gives up a little on the training set (R² = 0.994) and does better on the test set (R² = 0.509). Penalizing the coefficients made the model generalize better. We will see that Lasso can do even better on this dataset.

### How can we overcome this difficulty?

#### Variable selection problem

As we have seen, the Ridge model works well when some explanatory variables are not very informative, because it shrinks their coefficients. But sometimes the true value of many coefficients is 0, i.e. the corresponding explanatory variables have no influence on the target variable. Ridge does not penalize these uninformative variables enough: it shrinks their coefficients, but never sets them exactly to zero. So Ridge cannot be used for variable selection.

That is why another model, called Lasso, exists: it selects the relevant variables and sets the coefficients of the useless variables to zero.

### The Lasso model

#### Intuition

The key intuition behind the Lasso model is that many of the available explanatory variables have no influence on the target variable. Their coefficients would therefore have a true value of 0 in the linear model.

Let $n$ be the number of observations, $p$ the number of explanatory variables and $s$ the number of relevant explanatory variables. Mathematically, this intuition (called the **sparsity** hypothesis) is written as follows:

$$
s<<n<<p
$$

> 👋 $<<$ means *much less than*

By keeping only the $s$ relevant variables, we hope to end up in a situation where the linear model can be applied without difficulty.

Mathematically, we wish to find $\beta$ that minimizes the following cost function:

$$
\widehat{\beta_n}=arg\min_{\beta:||\beta||_{0}\leq{s}}||Y-X^{t}\beta||^{2}_{2}
$$

This equation means that $\widehat{\beta_n}$ is the vector that minimizes $||Y-X^{t}\beta||^{2}_{2}$, under the constraint that $\beta$ has at most $s$ non-zero elements.

Here, $||\:||_{0}$ is the "zero norm": it counts the number of non-zero elements in $\beta$.

However, this constraint is very hard to optimize, because it does not define a convex set. Instead, we use a weaker constraint that gives a convex cost function. It is based on the "L1 norm" $||\:||_{1}$, defined as the sum of the absolute values of the components of a vector.

#### Lasso estimator



::: {.callout-tip title="Intuition"}
Ridge and Lasso both pull the coefficients towards zero, but in a different way. Ridge's squared penalty becomes very weak near zero, so small coefficients are only gently shrunk and never quite reach 0. Lasso's absolute-value penalty pushes with the same force all the way down, so a coefficient that is not useful enough is pushed exactly to 0: the variable is removed from the model.
:::



Here we introduce the _Least Absolute Shrinkage and Selection Operator_ (LASSO), defined by the following cost function:

$$
||Y-X^{t}\beta||^{2}_{2}+\lambda||\beta||_{1}
$$

Here is what you need to know about the LASSO model:

* $\lambda$, the penalty constant, must be chosen carefully. In general, LASSO solvers use cross-validation (explained in the next lesson): they compute the estimator for many values of $\lambda$ to find the most relevant ones.
* The higher $\lambda$, the sparser the solution $\hat{\beta}_{LASSO}$ (i.e. it contains few non-zero elements), the higher the bias and the lower the variance.
* The smaller $\lambda$, the more non-zero coefficients. This decreases the bias of the model but can dramatically increase the variance (this is called overfitting).
* In practice, you will soon discover that the bias introduced by LASSO can be very large. If you need precise results, you can use LASSO to select the best variables, then fit a plain linear model on these variables only to remove this bias.

### See it with data 🔬: Lasso sets coefficients exactly to zero

Let's go back to our small dataset with 6 variables (only 3 useful) and draw the Lasso coefficient path, as we did for Ridge.

In [ ]:
alphas_lasso = np.logspace(-2, 2, 40)    # from 0.01 to 100
paths_lasso = pd.DataFrame([Lasso(alpha=a).fit(X, y).coef_ for a in alphas_lasso],
                           index=alphas_lasso, columns=[f"X{j}" for j in range(6)])

fig = px.line(paths_lasso, log_x=True, title="Lasso coefficient path",
              labels={"index": "λ (alpha, log scale)", "value": "coefficient", "variable": "variable"})
fig.show()

comparison = pd.DataFrame({"true": true_coef,
                           "Ridge (alpha=10)": Ridge(alpha=10).fit(X, y).coef_,
                           "Lasso (alpha=10)": Lasso(alpha=10).fit(X, y).coef_},
                          index=paths.columns).round(2)
comparison

,true,Ridge (alpha=10),Lasso (alpha=10)
X0,0.00,1.22,0.00
X1,81.91,69.71,71.38
X2,0.00,2.75,0.00
X3,98.24,83.82,89.16
X4,0.00,-3.27,-0.00
X5,31.06,28.41,22.79


**What we see:** on the Lasso path, the lines hit 0 one after the other and *stay* there. With the same λ = 10, Ridge keeps all 6 coefficients non-zero (for example 2.7 and -3.3 for two useless variables), while Lasso sets the useless variables to exactly 0 (shown as 0 or -0) and keeps only the 3 useful ones (71.4, 89.2 and 22.8, a bit smaller than the true values: this is the bias mentioned above).

Now the real test: the gene dataset. The researchers who built it also gave us the list of the 20 genes that truly drive the target (`gene_expression_true_genes.csv`). Can Lasso find them among 2,000?

In [ ]:
true_genes = set(pd.read_csv("../data/gene_expression_true_genes.csv")["gene"])

lasso = make_pipeline(StandardScaler(), Lasso(alpha=0.1, max_iter=10000)).fit(X_train, y_train)
selected = X_train.columns[lasso[-1].coef_ != 0]

print(f"Genes kept by Lasso: {len(selected)} out of {X_train.shape[1]}")
print(f"True genes among them: {len(true_genes & set(selected))} out of {len(true_genes)}")
print(f"R² test: {r2_score(y_test, lasso.predict(X_test)):.3f}")
ridge = models["Ridge (alpha=100)"]
print(f"For comparison, Ridge sets {np.sum(ridge[-1].coef_ == 0)} coefficients to exactly 0")

Genes kept by Lasso: 141 out of 2000
True genes among them: 19 out of 20
R² test: 0.740
For comparison, Ridge sets 7 coefficients to exactly 0


**What we see:** Lasso keeps only 141 genes out of 2,000, and 19 of the 20 true genes are among them. Its test R² (0.740) is much better than that of Ridge (0.509) and of the plain linear regression (0.409). Ridge, on the other hand, keeps (almost) every gene in the model: it sets only 7 coefficients to exactly 0, and only because these genes have the same value for all 200 training patients (a constant column carries no information at all). It shrinks, but it does not select. This is exactly the sparsity hypothesis at work: few variables matter, and Lasso finds them.

## Underfitting and Overfitting

Two notions were mentioned above. Here is what they mean:

* **Underfitting** happens when a model is too simple to give a good estimate of the target variable.
* **Overfitting** is the opposite: a model so complex that it fits the training data too well. Such a model is useless in practice, because it is very unlikely to generalize well to new, unseen data.

![under_over_fitting](../img/underfitting_overfitting.png)

The figure above shows, from left to right, underfitting, a good fit, and overfitting.

### See it with data 🔬: from underfitting to overfitting

We draw only 12 noisy training points from the curve $y = \sin(3x)$, fit polynomials of degree 1 to 11, and measure the error on the training points and on 200 new test points. The degree plays the role of model complexity.

In [ ]:
r = np.random.default_rng(0)
x_train = r.uniform(-1, 1, 12);  y_train_poly = np.sin(3 * x_train) + r.normal(0, 0.3, 12)
x_test = r.uniform(-1, 1, 200);  y_test_poly = np.sin(3 * x_test) + r.normal(0, 0.3, 200)

errors, fitted = [], {}
for degree in range(1, 12):
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    fitted[degree] = model.fit(x_train.reshape(-1, 1), y_train_poly)
    errors.append({"degree": degree,
                   "train MSE": mean_squared_error(y_train_poly, model.predict(x_train.reshape(-1, 1))),
                   "test MSE": mean_squared_error(y_test_poly, model.predict(x_test.reshape(-1, 1)))})
errors = pd.DataFrame(errors).set_index("degree")

fig = px.line(errors, markers=True, log_y=True, title="Training vs test error as the model gets more complex",
              labels={"value": "MSE (log scale)", "variable": ""})
fig.show()
errors.loc[[1, 3, 11]].round(3)

,train MSE,test MSE
degree,,
1,0.171,0.333
3,0.070,0.114
11,0.000,266.261


In [ ]:
fig = px.scatter(x=x_train, y=y_train_poly, title="Underfitting, good fit and overfitting",
                 labels={"x": "x", "y": "y"})
x_plot = np.linspace(-1, 1, 200)
for degree in [1, 3, 11]:
    fig.add_scatter(x=x_plot, y=fitted[degree].predict(x_plot.reshape(-1, 1)), mode="lines",
                    name=f"degree {degree}")
fig.update_yaxes(range=[-2.5, 2.5])
fig.show()

**What we see:** the training error goes down as the degree increases, until degree 11 reaches 0: with 12 coefficients for 12 points, the curve passes exactly through every training point. The test error tells another story. Degree 1 underfits (train MSE 0.171, test MSE 0.333), degree 3 is about right (test MSE 0.114, close to the noise level 0.3² = 0.09), and degree 11 overfits badly (test MSE around 266). The second plot shows the same three models: a straight line that misses the curve, a smooth curve, and a curve that wiggles wildly between the training points.

## Resources 📚📚

* [Ridge Regression and classification](https://scikit-learn.org/stable/modules/linear_model.html#ridge-regression-and-classification)

* [Lasso](https://scikit-learn.org/stable/modules/linear_model.html#lasso)

* [L1 and L2 Regularization](https://towardsdatascience.com/l1-and-l2-regularization-methods-ce25e7fc831c)

::: {.sb-next}
**[Next: Cross-validation and grid search →](../lessons/cross-validation-and-grid-search.ipynb)**
:::